In [2]:
import os

import ROOT


def compare(files, objNames, legendTexts, outputPath, suffix):
    colors = [
        # ROOT.kBlack,       
        ROOT.kRed-4,       
        ROOT.kBlue-4, 
        ROOT.kGreen+2,     
        ROOT.kOrange+7,   
        # ROOT.kYellow-7,    
        ROOT.kMagenta-3,   
        ROOT.kCyan-3,      
        ROOT.kSpring-5,    
        ROOT.kViolet-4,    
        ROOT.kTeal-5,    
        ROOT.kGray+1    
    ]
    markerStyles = [
        20,
        20,
        20,
        20,
        20,
        21,
        21,
        21,
        21,
        22,
    ]

    canvas = ROOT.TCanvas("c1", "c1", 1600, 1200)
    canvas.SetLeftMargin(0.12)
    canvas.SetBottomMargin(0.08)
    canvas.SetRightMargin(0.02)
    canvas.SetTopMargin(0.02)
    Legend = ROOT.TLegend(0.2, 0.75, 0.6, 0.95)
    Legend.SetBorderSize(0)
    Legend.SetFillStyle(0)
    legendFont = 42
    Legend.SetTextFont(legendFont)  
    Legend.SetTextSize(0.04)

    frame = canvas.DrawFrame(0, -0.01, 12, 0.18)
    frame.GetYaxis().SetTitle("D^{0} v_{2}")
    frame.GetXaxis().SetTitle("#it{p}_{T} (GeV/#it{c})")

    objs = []
    ratio_objs = []
    for i, fpath, objName, color, markerStyle in zip(range(len(files)), files, objNames, colors, markerStyles):
        f = ROOT.TFile.Open(fpath, "read")
        obj = f.Get(objName)
        obj.SetDirectory(0) if hasattr(obj, "SetDirectory") else None
        obj.SetLineColor(color)
        obj.SetLineWidth(4)
        obj.SetMarkerColor(color)
        obj.SetMarkerStyle(markerStyle)
        obj.SetMarkerSize(2)
        if files.index(fpath) == 0:
            # obj.SetTitle("Inclusive D^{0} v_{2} in OO collisions")
            # obj.Scale(1/0.07)
            obj.GetYaxis().SetTitle("D^{0} v_{2}")
            # obj.GetYaxis().SetRangeUser(0, 0.5)
            obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same")
        elif files.index(fpath) == len(files)-1:
            obj.SetLineColor(ROOT.kBlack)
            obj.SetLineWidth(4)
            obj.SetMarkerColor(ROOT.kBlack)
            obj.SetMarkerStyle(20)
            obj.SetMarkerSize(2)
            obj.Draw("same")
        else:
            # obj.SetTitle("")
            # obj.GetYaxis().SetTitle("inclusive D^{0} v_{2}")
            # obj.GetXaxis().SetTitle("p_{T} (GeV/c)")
            obj.Draw("same")
        Legend.AddEntry(obj, f"{legendTexts[i]}", "lp")
        objs.append(obj)
        f.Close()
    Legend.Draw()
    # canvas.Draw()
    canvas.Update()
    # the last one is the demestor, and the rest are the numerator, calculate and draw the ratio

    canvas_ratio = ROOT.TCanvas("c2", "c2", 1600, 1200)
    canvas_ratio.SetLeftMargin(0.12)
    canvas_ratio.SetBottomMargin(0.08)
    canvas_ratio.SetRightMargin(0.02)
    canvas_ratio.SetTopMargin(0.02)
    frame_ratio = canvas_ratio.DrawFrame(0, 0, 12, 2.1)
    for i in range(len(objs)-1):
        ratio = objs[i].Clone(f"ratio_{i}")
        nBins = ratio.GetNbinsX()
        nBins_den = objs[-1].GetNbinsX()
        if nBins != nBins_den:
            for b in range(1, ratio.GetNbinsX() + 1):
                num = objs[i].GetBinContent(b)
                den = objs[-1].GetBinContent(b)
                
                if den != 0:
                    temp_ratio = num / den
                    ratio.SetBinContent(b, num / den)
                    # num_err = objs[i].GetBinError(b)
                    # den_err = objs[-1].GetBinError(b)
                    # ratio_err = temp_ratio * ((num_err / num) ** 2 + (den_err / den) ** 2) ** 0.5 if num != 0 else 0
                    # ratio.SetBinError(b, ratio_err)
                else:
                    ratio.SetBinContent(b, 0)
                    ratio.SetBinError(b, 0)
        else:
            ratio.Divide(objs[i], objs[-1])
        ratio.SetLineColor(colors[i])
        ratio.SetLineWidth(4)
        ratio.SetMarkerColor(colors[i])
        ratio.SetMarkerStyle(markerStyles[i])
        ratio.SetMarkerSize(2)
        ratio.GetYaxis().SetTitle("Ratio to Biao sp prompt")
        ratio.GetXaxis().SetTitle("p_{T} (GeV/c)")
        ratio.Draw("same")
        ratio_objs.append(ratio)
    line = ROOT.TLine(0, 1, 12, 1)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kBlack)
    line.Draw("same")
    # canvas_ratio.Draw()
    canvas_ratio.Update()

    os.system(f"mkdir -p {outputPath}")
    ouput = outputPath + f"compare_{suffix}.root"
    canvas.SaveAs(outputPath + f"compare_{suffix}.png")
    canvas_ratio.SaveAs(outputPath + f"compare_ratio_{suffix}.png")
    outputfile = ROOT.TFile(ouput, "recreate")
    canvas.Write("c1")
    canvas_ratio.Write("c2")
    for obj in objs:
        obj.Write()
    for obj in ratio_objs:
        obj.Write()
    outputfile.Close()



In [30]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncSpline/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncRaw/final_results.root",
    
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncSpline/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncRaw/final_results.root",

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d4_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d6_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d8_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",

    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",

    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",

    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 Periodic Gaus LM",
    "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 Spline LM",
    "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 Raw LM",
    
    "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 Periodic Gaus LM",
    "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 Spline LM",
    "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 Raw LM",

    "2pc DeltaPhi 0.4<|#Delta#eta|<1.3",
    "2pc DeltaPhi 0.6<|#Delta#eta|<1.3",
    "2pc DeltaPhi 0.8<|#Delta#eta|<1.3",

    "Biao sp prompt",
]

suffix = "0d_AppDeltaPhi"
compare(files[0:3], objNames[0:3], legendTexts[0:3], outputPath, suffix)

suffix = "0d2_AppDeltaPhi"
compare(files[3:6], objNames[3:6], legendTexts[3:6], outputPath, suffix)

suffix = "etaVariation_AppDeltaPhi"
compare([files[0]] + [files[3]] + files[6:], [objNames[0]] + [objNames[3]] + objNames[6:], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:], outputPath, suffix)

suffix = "etaVariation_AppDeltaPhi_noSP"
compare([files[0]] + [files[3]] + files[6:9], [objNames[0]] + [objNames[3]] + objNames[6:9], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:9], outputPath, suffix)

suffix = "comparison_AppDeltaPhi_noSP_no0dAnd0d2"
compare(files[6:9], objNames[6:9], legendTexts[6:9], outputPath, suffix)

suffix = "comparison_AppDeltaPhi_no0dAnd0d2"
compare(files[6:], objNames[6:], legendTexts[6:], outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison/compare_0d_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison/compare_ratio_0d_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison/compare_0d2_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison/compare_ratio_0d2_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison/compare_etaVariation_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/D

In [36]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/comparison/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/CorrelExtract_0d_1d3_AppDeltaPhi/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncSpline/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncRaw/final_results.root",
    
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/CorrelExtract_0d2_1d3_AppDeltaPhi/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncSpline/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncRaw/final_results.root",

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/CorrelExtract_0d4_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/CorrelExtract_0d6_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/CorrelExtract_0d8_1d3_AppDeltaPhi/final_results.root",

    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",

    "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",

    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",

    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 Periodic Gaus LM",
    # "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 Spline LM",
    # "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 Raw LM",
    
    "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 Periodic Gaus LM",
    # "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 Spline LM",
    # "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 Raw LM",

    "2pc DeltaPhi 0.4<|#Delta#eta|<1.3",
    "2pc DeltaPhi 0.6<|#Delta#eta|<1.3",
    "2pc DeltaPhi 0.8<|#Delta#eta|<1.3",

    "Biao sp prompt",
]

suffix = "AppDeltaPhi_32bins_16binsLM_noSP"
compare(files[0:], objNames[0:], legendTexts[0:], outputPath, suffix)

# suffix = "0d_AppDeltaPhi"
# compare(files[0:3], objNames[0:3], legendTexts[0:3], outputPath, suffix)

# suffix = "0d2_AppDeltaPhi"
# compare(files[3:6], objNames[3:6], legendTexts[3:6], outputPath, suffix)

# suffix = "etaVariation_AppDeltaPhi"
# compare([files[0]] + [files[3]] + files[6:], [objNames[0]] + [objNames[3]] + objNames[6:], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:], outputPath, suffix)

# suffix = "etaVariation_AppDeltaPhi_noSP"
# compare([files[0]] + [files[3]] + files[6:9], [objNames[0]] + [objNames[3]] + objNames[6:9], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:9], outputPath, suffix)

# suffix = "comparison_AppDeltaPhi_noSP_no0dAnd0d2"
# compare(files[6:9], objNames[6:9], legendTexts[6:9], outputPath, suffix)

# suffix = "comparison_AppDeltaPhi_no0dAnd0d2"
# compare(files[6:], objNames[6:], legendTexts[6:], outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/comparison/compare_AppDeltaPhi_32bins_16binsLM_noSP.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation_32bins/comparison/compare_ratio_AppDeltaPhi_32bins_16binsLM_noSP.png has been created


In [8]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_mass/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppMass/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppMass/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d4_1d3_AppMass/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d6_1d3_AppMass/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d8_1d3_AppMass/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc Mass 0.0<|#Delta#eta|<1.3 Periodic LM",
    
    "2pc Mass 0.2<|#Delta#eta|<1.3 Periodic LM",

    "2pc Mass 0.4<|#Delta#eta|<1.3 Periodic LM",
    "2pc Mass 0.6<|#Delta#eta|<1.3 Periodic LM",
    "2pc Mass 0.8<|#Delta#eta|<1.3 Periodic LM",

    "Biao sp prompt",
]

suffix = "free_massdiffRYandLM_AppMass"
compare(files[0:], objNames[0:], legendTexts[0:], outputPath, suffix)

# suffix = "0d_AppDeltaPhi"
# compare(files[0:3], objNames[0:3], legendTexts[0:3], outputPath, suffix)

# suffix = "0d2_AppDeltaPhi"
# compare(files[3:6], objNames[3:6], legendTexts[3:6], outputPath, suffix)

# suffix = "etaVariation_AppDeltaPhi"
# compare([files[0]] + [files[3]] + files[6:], [objNames[0]] + [objNames[3]] + objNames[6:], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:], outputPath, suffix)

# suffix = "etaVariation_AppDeltaPhi_noSP"
# compare([files[0]] + [files[3]] + files[6:9], [objNames[0]] + [objNames[3]] + objNames[6:9], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:9], outputPath, suffix)

# suffix = "comparison_AppDeltaPhi_noSP_no0dAnd0d2"
# compare(files[6:9], objNames[6:9], legendTexts[6:9], outputPath, suffix)

# suffix = "comparison_AppDeltaPhi_no0dAnd0d2"
# compare(files[6:], objNames[6:], legendTexts[6:], outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_mass/compare_free_massdiffRYandLM_AppMass.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_mass/compare_ratio_free_massdiffRYandLM_AppMass.png has been created


In [42]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_mass/"
files = [
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppMass/results_free_noBL_tempFuncGausPeriodic/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppMass/results_free_noBL_tempFuncGausPeriodic/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d4_1d3_AppMass/results_fixed_noBL_tempFuncGausPeriodic/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d6_1d3_AppMass/results_fixed_noBL_tempFuncGausPeriodic/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d8_1d3_AppMass/results_fixed_noBL_tempFuncGausPeriodic/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    # "hVnSimFit",
    # "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hV2VsPtPrompt",
]
legendTexts = [
    # "2pc Mass 0.0<|#Delta#eta|<1.3 Periodic Gaus LM",
    
    # "2pc Mass 0.2<|#Delta#eta|<1.3 Periodic Gaus LM",

    "2pc Mass fixed 0.4<|#Delta#eta|<1.3",
    "2pc Mass fixed 0.6<|#Delta#eta|<1.3",
    "2pc Mass fixed 0.8<|#Delta#eta|<1.3",

    "Biao sp prompt",
]

suffix = "AppMass"
compare(files[0:], objNames[0:], legendTexts[0:], outputPath, suffix)

# suffix = "0d_AppDeltaPhi"
# compare(files[0:3], objNames[0:3], legendTexts[0:3], outputPath, suffix)

# suffix = "0d2_AppDeltaPhi"
# compare(files[3:6], objNames[3:6], legendTexts[3:6], outputPath, suffix)

# suffix = "etaVariation_AppDeltaPhi"
# compare([files[0]] + [files[3]] + files[6:], [objNames[0]] + [objNames[3]] + objNames[6:], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:], outputPath, suffix)

# suffix = "etaVariation_AppDeltaPhi_noSP"
# compare([files[0]] + [files[3]] + files[6:9], [objNames[0]] + [objNames[3]] + objNames[6:9], [legendTexts[0]] + [legendTexts[3]] + legendTexts[6:9], outputPath, suffix)

# suffix = "comparison_AppDeltaPhi_noSP_no0dAnd0d2"
# compare(files[6:9], objNames[6:9], legendTexts[6:9], outputPath, suffix)

# suffix = "comparison_AppDeltaPhi_no0dAnd0d2"
# compare(files[6:], objNames[6:], legendTexts[6:], outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_mass/compare_AppMass.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_mass/compare_ratio_AppMass.png has been created


In [24]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_tempFunc/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncSpline/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncSpline/final_results.root"
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d4_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d6_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/CorrelExtract_0d8_1d3_AppDeltaPhi/results_fixed_noBL_tempFuncGausPeriodic/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 GausPeriodic",
    # "2pc DeltaPhi 0.0<|#Delta#eta|<1.3 spline",
    "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 GausPeriodic",
    # "2pc DeltaPhi 0.2<|#Delta#eta|<1.3 spline",
    "2pc DeltaPhi 0.4<|#Delta#eta|<1.3",
    "2pc DeltaPhi 0.6<|#Delta#eta|<1.3",
    "2pc DeltaPhi 0.8<|#Delta#eta|<1.3",
    "Biao sp prompt",
]
suffix = "0d_0d2_etaVariation_AppDeltaPhi_noSP"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_tempFunc/compare_0d_0d2_etaVariation_AppDeltaPhi_noSP.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020_gausPer/etaVariation/comparison/v2_comparison_tempFunc/compare_ratio_0d_0d2_etaVariation_AppDeltaPhi_noSP.png has been created


In [37]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/compare/"
files = [

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/CorrelExtract_0d8_1d3_normalized/final_results_NFsub_Wped.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/CorrelExtract_0d8_1d3_normalized/final_results_NFsub_freeLM.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/CorrelExtract_0d8_1d3_normalized/final_results_NFsub_fixedLM.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/CorrelExtract_0d8_1d3_woNF/final_results.root",

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/CorrelExtract_0d8_1d3_mass/CorrelationFitResults_wPed_freeLM/ry/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/CorrelExtract_0d8_1d3_mass/CorrelationFitResults_woPed_freeLM/ry/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/CorrelExtract_0d8_1d3_mass/CorrelationFitResults_woNFSub/ry/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hvnSimFit",
    "hvnSimFit",
    "hvnSimFit",
    # "hvnSimFit",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [

    "2pc deltaPhi w pedestal free LM",
    "2pc deltaPhi w/o pedestal free LM",
    "2pc deltaPhi w/o pedestal fixed LM",
    "2pc deltaPhi w/o NF sub.",

    "2pc mass w pedestal free LM",
    "2pc mass w/o pedestal free LM",
    "2pc mass w/o NF sub.",
    "Biao sp prompt",
]

# suffix = "0d8_1d3_roofit_deltaPhi_wrtNF"
# compare(files, objNames, legendTexts, outputPath, suffix)
suffix = "0d8_1d3_roofit_wPed_freeLM"
compare([files[0], files[4], files[7]], [objNames[0], objNames[4], objNames[7]], [legendTexts[0], legendTexts[4], legendTexts[7]], outputPath, suffix)

suffix = "0d8_1d3_roofit_woPed_freeLM"
compare([files[1], files[5], files[7]], [objNames[1], objNames[5], objNames[7]], [legendTexts[1], legendTexts[5], legendTexts[7]], outputPath, suffix)

suffix = "0d8_1d3_roofit_woNFSub"
compare([files[3], files[6], files[7]], [objNames[3], objNames[6], objNames[7]], [legendTexts[3], legendTexts[6], legendTexts[7]], outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/compare/compare_0d8_1d3_roofit_wPed_freeLM.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/compare/compare_ratio_0d8_1d3_roofit_wPed_freeLM.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/compare/compare_0d8_1d3_roofit_woPed_freeLM.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/compare/compare_ratio_0d8_1d3_roofit_woPed_freeLM.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/compare/compare_0d8_1d3_roofit_woNFSub.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/firstlook/compare/compare_ratio_0d8_1d3_roofit_woNFSub.p

In [ ]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_wo_NF_sub/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_fixed_woBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_free_woBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc mass w/o NF sub.",
    "2pc mass fixed LM w/o BL",
    "2pc mass free LM w/o BL",
    "Biao sp prompt",
]
suffix = "fitprocedure_AppMass"
compare(files, objNames, legendTexts, outputPath, suffix)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/compare_fitprocedure_AppMass.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/compare_ratio_fitprocedure_AppMass.png has been created


In [15]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_wo_NF_sub/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_free_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc DeltaPhi w/o NF sub.",
    "2pc DeltaPhi fixed LM w/o BL",
    "2pc DeltaPhi free LM w/o BL",
    "Biao sp prompt",
]
suffix = "fitprocedure_AppDeltaPhi"
compare(files, objNames, legendTexts, outputPath, suffix)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/compare_fitprocedure_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/compare/compare_ratio_fitprocedure_AppDeltaPhi.png has been created


In [32]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/"
files = [

    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d_1d3_AppDeltaPhi/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d2_1d3_AppDeltaPhi/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d4_1d3_AppDeltaPhi/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d6_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k70100/CorrelExtract_0d4_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d4_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [
    # "2pc DeltaPhi 0.-1.3",
    # "2pc DeltaPhi 0.2-1.3",
    "2pc DeltaPhi 0.4-1.3 60-100% for LM",
    "2pc DeltaPhi 0.4-1.3 70-100% for LM",
    "2pc DeltaPhi 0.4-1.3 80-100% for LM",
    "Biao sp prompt",
]
suffix = "centVariation_0d4_AppDeltaPhi"
compare(files, objNames, legendTexts, outputPath, suffix)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/compare_centVariation_0d4_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/compare_ratio_centVariation_0d4_AppDeltaPhi.png has been created


In [27]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/"
files = [

    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d_1d3_AppDeltaPhi/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d2_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d4_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d6_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d8_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [
    # "2pc DeltaPhi 0.-1.3",
    # "2pc DeltaPhi 0.2-1.3",
    "2pc DeltaPhi 0.4-1.3",
    "2pc DeltaPhi 0.6-1.3",
    "2pc DeltaPhi 0.8-1.3",
    "Biao sp prompt",
]
suffix = "k80100_0d4_etaGapVariation_AppDeltaPhi"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/compare_k80100_0d4_etaGapVariation_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/compare_ratio_k80100_0d4_etaGapVariation_AppDeltaPhi.png has been created


In [26]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/"
files = [

    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k70100/CorrelExtract_0d_1d3_AppDeltaPhi/final_results.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k70100/CorrelExtract_0d2_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k70100/CorrelExtract_0d4_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k70100/CorrelExtract_0d6_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k70100/CorrelExtract_0d8_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    # "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [
    # "2pc DeltaPhi 0.-1.3",
    # "2pc DeltaPhi 0.2-1.3",
    "2pc DeltaPhi 0.4-1.3",
    "2pc DeltaPhi 0.6-1.3",
    "2pc DeltaPhi 0.8-1.3",
    "Biao sp prompt",
]
suffix = "k70100_0d4_etaGapVariation_AppDeltaPhi"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/compare_k70100_0d4_etaGapVariation_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/cent/compare_ratio_k70100_0d4_etaGapVariation_AppDeltaPhi.png has been created


In [ ]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc DeltaPhi 0.-1.3",
    "2pc DeltaPhi 0.2-1.3",
    "2pc DeltaPhi 0.4-1.3",
    "2pc DeltaPhi 0.6-1.3",
    "2pc DeltaPhi 0.8-1.3",
    "Biao sp prompt",
]
suffix = "etaGapVariation_AppDeltaPhi"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_etaGapVariation_AppDeltaPhi.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ratio_etaGapVariation_AppDeltaPhi.png has been created


In [34]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [

    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_free_woBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc mass free LM 0.-1.3",
    "2pc mass free LM 0.2-1.3",
    "2pc mass free LM 0.4-1.3",
    "2pc mass free LM 0.6-1.3",
    "2pc mass free LM 0.8-1.3",
    "Biao sp prompt",
]
suffix = "etaGapVariatio_freeLm_AppMass"
compare(files, objNames, legendTexts, outputPath, suffix)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_etaGapVariatio_freeLm_AppMass.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ratio_etaGapVariatio_freeLm_AppMass.png has been created


In [21]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_AppDeltaPhi/results_free_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_AppDeltaPhi/results_free_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_AppDeltaPhi/results_free_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_AppDeltaPhi/results_free_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_free_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc DeltaPhi 0.-1.3",
    "2pc DeltaPhi 0.2-1.3",
    "2pc DeltaPhi 0.4-1.3",
    "2pc DeltaPhi 0.6-1.3",
    "2pc DeltaPhi 0.8-1.3",
    "Biao sp prompt",
]
suffix = "etaGapVariation_AppDeltaPhi_free"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_etaGapVariation_AppDeltaPhi_free.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ratio_etaGapVariation_AppDeltaPhi_free.png has been created


In [18]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_free_woBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc mass 0.-1.3",
    "2pc mass 0.2-1.3",
    "2pc mass 0.4-1.3",
    "2pc mass 0.6-1.3",
    "2pc mass 0.8-1.3",
    "Biao sp prompt",
]
suffix = "etaGapVariation_AppMass_free"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_etaGapVariation_AppMass_free.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ratio_etaGapVariation_AppMass_free.png has been created


In [19]:
outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d_1d3_Appmass/results_limited_F/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d2_1d3_Appmass/results_limited_F/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d4_1d3_Appmass/results_limited_F/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_Appmass/results_limited_F/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_limited_F/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hVnSimFit",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc mass 0.-1.3",
    "2pc mass 0.2-1.3",
    "2pc mass 0.4-1.3",
    "2pc mass 0.6-1.3",
    "2pc mass 0.8-1.3",
    "Biao sp prompt",
]
suffix = "etaGapVariation_AppMass_limitedF"
compare(files, objNames, legendTexts, outputPath, suffix)

Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_etaGapVariation_AppMass_limitedF.png has been created
Info in <TCanvas::Print>: png file /home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/compare_ratio_etaGapVariation_AppMass_limitedF.png has been created


In [1]:
# 2pc deltaphi fixed 0.8-1.3 in 60-100
# 2pc deltaphi fixed 0.8-1.3 in 70-100
# 2pc deltaphi fixed 0.4-1.3 in 80-100
# 2pc mass fixed 0.8-1.3 in 60-100
# 2pc mass free 0.6-1.3 in 60-100
# sp

outputPath = "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/compare/"
files = [
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_AppDeltaPhi/results_fixed_noBL/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k70100/CorrelExtract_0d8_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/k80100/CorrelExtract_0d6_1d3_AppDeltaPhi/final_results.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/fitprocedure/CorrelExtract_0d8_1d3_Appmass/results_free_woBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fifth/k020/etavariation/CorrelExtract_0d6_1d3_Appmass/results_free_noBL/CorrelationFitResults/raw_yields/InvMassVsV2_PtAssoc02to30.root",
    "/home/wuct/MetaData/DATA/OO/apass2/corr/results/fthlook_Biao/large/old/k020/v2VsFracD0020Biao.root",
    # "/home/wuct/MetaData/DATA/OO/apass2/sp/result/v2_OO_pt_cms_pbpb_6080.root",
]
objNames = [
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hv2Delta_PtBinAssoc1_InvMassBin1",
    "hVnSimFit",
    "hVnSimFit",
    "hV2VsPtPrompt",
]
legendTexts = [
    "2pc #Delta#phi 0.8<|#Delta#eta|<1.3 fixed LM in 60-100% c.c.",
    "2pc #Delta#phi 0.8<|#Delta#eta|<1.3 fixed LM in 70-100% c.c.",
    "2pc #Delta#phi 0.6<|#Delta#eta|<1.3 fixed LM in 80-100% c.c.",
    "2pc mass 0.8<|#Delta#eta|<1.3 free LM in 60-100% c.c.",
    "2pc mass 0.6<|#Delta#eta|<1.3 free LM in 60-100% c.c.",
    "Biao sp prompt",
]
suffix = "final_0d8_1d3_AppDeltaPhi_mass_comparison"
compare(files, objNames, legendTexts, outputPath, suffix)

NameError: name 'compare' is not defined